# LOAD in MPDAG: Real Dataset Applications

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%reload_ext autoreload

## Standard libraries

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx

from collections import defaultdict
from itertools import chain, combinations
from typing import Callable, NamedTuple, Sequence

import json
from pathlib import Path
import pickle

import numpy as np
import networkx as nx
from rpy2.robjects import r

from time import perf_counter
from sklearn.preprocessing import StandardScaler

from causallearn.utils.cit import CIT
from causallearn.utils.Dataset import load_dataset

from time import perf_counter
from tqdm import tqdm
from sklearn.linear_model import LinearRegression
from src.real_data_applications.sachs import fit_oracle_weights
from src.real_data_applications.sachs import get_protein_name, get_sachs_ground_truth_dag
from src.real_data_applications.sachs import forward_pairs_idx
from src.running import run_experiment_mpdag_real

## R libraries

In [ ]:
import rpy2.robjects as robjects
import os



try:
    robjects.r('library(pcalg)')
    robjects.r('library(igraph)')
    robjects.r('library(bnlearn)')
    print("✅ R environment for LOAD is ready!")
except Exception as e:
    print(f"❌ Failed to load library: {e}")


## Util functions

In [ ]:
from src.utils import draw_ground_truth_from_dag

# Experiments

## Exp 1: application to Sachs data only global related BK

### Data loading

In [ ]:
from src.real_data_applications.sachs import get_protein_name, get_sachs_ground_truth_dag
true_dag_sachs = get_sachs_ground_truth_dag()
from src.real_data_applications.sachs import forward_pairs_idx
sachs_data, sachs_labels = load_dataset("sachs")
from src.running import run_experiment_mpdag_real

In [ ]:
sachs_data

In [ ]:
draw_ground_truth_from_dag(true_dag_sachs)

### Running

#### Test run multiple relations



In [ ]:

true_dag_sachs = get_sachs_ground_truth_dag()
sachs_data, sachs_labels = load_dataset("sachs")

base_args_skeleton = {
      "data_name": "sachs",
      "real_data": sachs_data,
      "true_dag": true_dag_sachs,
      "observed": sachs_data.shape[1],
      "alpha": 0.01,
      "ci_test": "fisherz",
}

base_seed = 42
frac_forb = 0
frac_req = 0.3

base_args_skeleton["fraction_forbidden"] = frac_forb
base_args_skeleton["fraction_required"] = frac_req

# Run experiment
result_dict_42, details_dict_42 = run_experiment_mpdag_real(base_args_skeleton, base_seed, forward_pairs_idx)

In [ ]:
import pandas as pd
import numpy as np

# 1. Define your relations in the same order they were generated
relations = [
    "Plcg -> PIP3", "Plcg -> PIP2", "Plcg -> PKC", "Plcg -> PKA", "Plcg -> Raf",
    "Plcg -> Mek", "Plcg -> Erk", "Plcg -> Akt", "Plcg -> P38", "Plcg -> Jnk",
    "PIP3 -> PIP2", "PIP3 -> PKC", "PIP3 -> PKA", "PIP3 -> Raf", "PIP3 -> Mek",
    "PIP3 -> Erk", "PIP3 -> Akt", "PIP3 -> P38", "PIP3 -> Jnk", "PIP2 -> PKC",
    "PIP2 -> PKA", "PIP2 -> Raf", "PIP2 -> Mek", "PIP2 -> Erk", "PIP2 -> Akt",
    "PIP2 -> P38", "PIP2 -> Jnk", "PKC -> PKA", "PKC -> Raf", "PKC -> Mek",
    "PKC -> Erk", "PKC -> Akt", "PKC -> P38", "PKC -> Jnk", "PKA -> Raf",
    "PKA -> Mek", "PKA -> Erk", "PKA -> Akt", "PKA -> P38", "PKA -> Jnk",
    "Raf -> Mek", "Raf -> Erk", "Raf -> Akt", "Mek -> Erk", "Mek -> Akt",
    "Erk -> Akt"
]

# 2. Build a list of records for the DataFrame
data = []
for i, rel in enumerate(relations):
    data.append({
        "Relation": rel,
        "Prec (With B)": details_dict_42['with_b']['precision'][i],
        "Rec (With B)": details_dict_42['with_b']['recall'][i],
        "F1 (With B)": details_dict_42['with_b']['f1'][i],
        "Prec (No B)": details_dict_42['without_b']['precision'][i],
        "Rec (No B)": details_dict_42['without_b']['recall'][i],
        "F1 (No B)": details_dict_42['without_b']['f1'][i],
    })

# 3. Create DataFrame
results_df = pd.DataFrame(data)

# Optional: Set Relation as index and format numbers for better readability
results_df.set_index("Relation", inplace=True)
pd.options.display.float_format = '{:,.3f}'.format

# 4. Filter for only "successful" relations (where With B > 0) to see what worked
successful_results = results_df[results_df["F1 (With B)"] > 0]

print("--- Full Results Table ---")
print(results_df)

print("\n--- Relations where 'With B' found a valid OSET ---")
print(successful_results)

In [ ]:
successful_results

#### Resolving issues on intervention distance

In [ ]:
from src.real_data_applications.sachs import fit_oracle_weights


true_dag_sachs = get_sachs_ground_truth_dag()
sachs_data, sachs_labels = load_dataset("sachs")
oracle_dag = fit_oracle_weights(true_dag_sachs, sachs_data)

base_args_skeleton = {
      "data_name": "sachs",
      "real_data": sachs_data,
      "true_dag": oracle_dag,
      "observed": sachs_data.shape[1],
      "alpha": 0.01,
      "ci_test": "fisherz",
      "sampling_strategy": "global"
}

base_seed = 42
frac_forb = 0
frac_req = 0.3

base_args_skeleton["fraction_forbidden"] = frac_forb
base_args_skeleton["fraction_required"] = frac_req

# Run experiment
result_dict_42, details_dict_42 = run_experiment_mpdag_real(base_args_skeleton, base_seed, forward_pairs_idx)

In [ ]:
sachs_data

In [ ]:
result_dict_42


## Exp 3: application to DREAM4 datasets

In [ ]:
from src.real_data_applications.sachs import fit_oracle_weights


### Subset 1

In [ ]:
goldstandard_path = "experiments/dream4/DREAM4_GoldStandard_InSilico_Size100_multifactorial_1.tsv"
dream4_net1_gs = pd.read_csv(
    goldstandard_path,
    sep='\t',
    header=None,
    names=['cause', 'outcome', 'direction']
)

dream4_net1_gs.head()

In [ ]:
dream4_net1_gs_filtered = dream4_net1_gs[dream4_net1_gs['direction'] == 1]

In [ ]:
dream4_net1_gs_filtered

In [ ]:
import networkx as nx

def sanitize_to_dag(graph: nx.DiGraph) -> nx.DiGraph:
    """
    Iteratively finds and breaks cycles in a directed graph to guarantee a strict DAG.
    It does this by removing one edge from every cycle it finds.
    """
    # Create a copy so we don't mutate the original biological graph
    safe_dag = graph.copy()
    
    cycle_count = 0
    while True:
        try:
            # Look for a cycle
            cycle = nx.find_cycle(safe_dag)
            
            # cycle is a list of edges, e.g., [(5, 51), (51, 5)]
            # We break the cycle by removing the very first edge in the list
            edge_to_remove = cycle[0]
            safe_dag.remove_edge(*edge_to_remove)
            
            cycle_count += 1
            print(f"Broke cycle by removing edge: {edge_to_remove}")
            
        except nx.NetworkXNoCycle:
            # If no cycles are found, we successfully made a DAG!
            break
            
    print(f"Sanitization complete. Removed {cycle_count} edges to ensure acyclicity.")
    return safe_dag

In [ ]:
import networkx as nx
import pandas as pd

# The order of nodes must perfectly match the columns of your multifactorial data.
# E.g., NODE_ORDER = list(multifactorial_data.columns)  # ['G1', 'G2', ..., 'G100']

def get_dream4_ground_truth_dag(gs_dataframe: pd.DataFrame, node_order: list) -> nx.DiGraph:
    """
    Constructs the ground truth DAG for the DREAM4 dataset.
    Nodes are integer indices corresponding to the order in node_order.
    """
    # 1. Create the reverse mapping (string name -> integer index)
    name_to_idx = {name: idx for idx, name in enumerate(node_order)}

    true_dag = nx.DiGraph()

    # 2. Add ALL nodes as integer indices (0 through len(node_order)-1)
    # This ensures disconnected genes still exist mathematically in your graph
    true_dag.add_nodes_from(range(len(node_order)))

    # 3. Extract true edges and map string names to integer indices
    edges_by_index = []
    
    # We assume your dataframe was loaded with columns: ['cause', 'outcome', 'edge']
    for _, row in gs_dataframe.iterrows():
        cause_name = row['cause']
        outcome_name = row['outcome']
        edge_exists = row['edge'] 
        
        # In DREAM4, the file lists non-existent edges as 0. We ONLY want the 1s.
        if edge_exists == 1:
            # Safety check to ensure the nodes exist in our multifactorial dataset
            if cause_name in name_to_idx and outcome_name in name_to_idx:
                u = name_to_idx[cause_name]
                v = name_to_idx[outcome_name]
                edges_by_index.append((u, v))

    # 4. Add the integer edges to the graph
    true_dag.add_edges_from(edges_by_index)

    return true_dag

# --- How to use it in your pipeline ---

# 1. Load your multifactorial data
multifactorial_df = pd.read_csv("../experiments/dream4/insilico_size100_1_multifactorial.tsv", sep='\t')
NODE_ORDER = list(multifactorial_df.columns)

# 2. Load the DREAM4 Gold Standard and give it headers
dream4_gs = pd.read_csv("../experiments/dream4/DREAM4_GoldStandard_InSilico_Size100_multifactorial_1.tsv", 
                        sep='\t', header=None, names=['cause', 'outcome', 'edge'])

# 3. Generate the true binary DAG
binary_dag = sanitize_to_dag(get_dream4_ground_truth_dag(dream4_gs, NODE_ORDER))

# 4. Fit the Oracle Weights to get your linear SEM
dream4_data_array = multifactorial_df.to_numpy()
oracle_dag = fit_oracle_weights(binary_dag, dream4_data_array)

In [ ]:
draw_ground_truth_from_dag(binary_dag)

In [ ]:
import networkx as nx

# 1. Get all mathematically identifiable forward pairs (Ancestor -> Descendant)
# This uses the true_dag you generated earlier
forward_pairs_idx = []
for node in binary_dag.nodes():
    # nx.descendants finds all nodes downstream (direct and indirect)
    for descendant in nx.descendants(binary_dag, node):
        forward_pairs_idx.append([node, descendant])

In [ ]:
len(forward_pairs_idx)

In [ ]:

base_args_skeleton = {
      "data_name": "dream5_net2",
      "real_data": dream4_data_array,
      "true_dag": oracle_dag,
      "observed": dream4_data_array.shape[1],
      "alpha": 0.01,
      "ci_test": "fisherz",
      "sampling_strategy": "local"
}

base_seed = 42
frac_forb = 0
frac_req = 0.3

base_args_skeleton["fraction_forbidden"] = frac_forb
base_args_skeleton["fraction_required"] = frac_req

# Run experiment
result_dict_dream4, details_dict_dream4 = run_experiment_mpdag_real(base_args_skeleton, base_seed, forward_pairs_idx)

In [ ]:
result_dict_dream4

### Generic pipeline

In [ ]:
subset_number = 1  # Change this to 2, 3, or 4 for the other DREAM4 networks

# 1. Load your multifactorial data
multifactorial_df = pd.read_csv(f"../experiments/dream4/insilico_size100_{subset_number}_multifactorial.tsv", sep='\t')
NODE_ORDER = list(multifactorial_df.columns)

# 2. Load the DREAM4 Gold Standard and give it headers
dream4_gs = pd.read_csv(f"../experiments/dream4/DREAM4_GoldStandard_InSilico_Size100_multifactorial_{subset_number}.tsv", 
                        sep='\t', header=None, names=['cause', 'outcome', 'edge'])

# 3. Generate the true binary DAG
binary_dag = sanitize_to_dag(get_dream4_ground_truth_dag(dream4_gs, NODE_ORDER))

# 4. Fit the Oracle Weights to get your linear SEM
dream4_data_array = multifactorial_df.to_numpy()
oracle_dag = fit_oracle_weights(binary_dag, dream4_data_array)

forward_pairs_idx = []
for node in binary_dag.nodes():
    # nx.descendants finds all nodes downstream (direct and indirect)
    for descendant in nx.descendants(binary_dag, node):
        forward_pairs_idx.append([node, descendant])

print(f"Number of forward pairs in DREAM4 subset {subset_number}: {len(forward_pairs_idx)}")

In [ ]:
subset_number = 2  # Change this to 2, 3, or 4 for the other DREAM4 networks

# 1. Load your multifactorial data
multifactorial_df = pd.read_csv(f"../experiments/dream4/insilico_size100_{subset_number}_multifactorial.tsv", sep='\t')
NODE_ORDER = list(multifactorial_df.columns)

# 2. Load the DREAM4 Gold Standard and give it headers
dream4_gs = pd.read_csv(f"../experiments/dream4/DREAM4_GoldStandard_InSilico_Size100_multifactorial_{subset_number}.tsv", 
                        sep='\t', header=None, names=['cause', 'outcome', 'edge'])

# 3. Generate the true binary DAG
binary_dag = sanitize_to_dag(get_dream4_ground_truth_dag(dream4_gs, NODE_ORDER))

# 4. Fit the Oracle Weights to get your linear SEM
dream4_data_array = multifactorial_df.to_numpy()
oracle_dag = fit_oracle_weights(binary_dag, dream4_data_array)

forward_pairs_idx = []
for node in binary_dag.nodes():
    # nx.descendants finds all nodes downstream (direct and indirect)
    for descendant in nx.descendants(binary_dag, node):
        forward_pairs_idx.append([node, descendant])

print(f"Number of forward pairs in DREAM4 subset {subset_number}: {len(forward_pairs_idx)}")

In [ ]:
subset_number = 3  # Change this to 2, 3, or 4 for the other DREAM4 networks

# 1. Load your multifactorial data
multifactorial_df = pd.read_csv(f"../experiments/dream4/insilico_size100_{subset_number}_multifactorial.tsv", sep='\t')
NODE_ORDER = list(multifactorial_df.columns)

# 2. Load the DREAM4 Gold Standard and give it headers
dream4_gs = pd.read_csv(f"../experiments/dream4/DREAM4_GoldStandard_InSilico_Size100_multifactorial_{subset_number}.tsv", 
                        sep='\t', header=None, names=['cause', 'outcome', 'edge'])

# 3. Generate the true binary DAG
binary_dag = sanitize_to_dag(get_dream4_ground_truth_dag(dream4_gs, NODE_ORDER))

# 4. Fit the Oracle Weights to get your linear SEM
dream4_data_array = multifactorial_df.to_numpy()
oracle_dag = fit_oracle_weights(binary_dag, dream4_data_array)

forward_pairs_idx = []
for node in binary_dag.nodes():
    # nx.descendants finds all nodes downstream (direct and indirect)
    for descendant in nx.descendants(binary_dag, node):
        forward_pairs_idx.append([node, descendant])

print(f"Number of forward pairs in DREAM4 subset {subset_number}: {len(forward_pairs_idx)}")

In [ ]:
subset_number = 4  # Change this to 2, 3, or 4 for the other DREAM4 networks

# 1. Load your multifactorial data
multifactorial_df = pd.read_csv(f"../experiments/dream4/insilico_size100_{subset_number}_multifactorial.tsv", sep='\t')
NODE_ORDER = list(multifactorial_df.columns)

# 2. Load the DREAM4 Gold Standard and give it headers
dream4_gs = pd.read_csv(f"../experiments/dream4/DREAM4_GoldStandard_InSilico_Size100_multifactorial_{subset_number}.tsv", 
                        sep='\t', header=None, names=['cause', 'outcome', 'edge'])

# 3. Generate the true binary DAG
binary_dag = sanitize_to_dag(get_dream4_ground_truth_dag(dream4_gs, NODE_ORDER))

# 4. Fit the Oracle Weights to get your linear SEM
dream4_data_array = multifactorial_df.to_numpy()
oracle_dag = fit_oracle_weights(binary_dag, dream4_data_array)

forward_pairs_idx = []
for node in binary_dag.nodes():
    # nx.descendants finds all nodes downstream (direct and indirect)
    for descendant in nx.descendants(binary_dag, node):
        forward_pairs_idx.append([node, descendant])

print(f"Number of forward pairs in DREAM4 subset {subset_number}: {len(forward_pairs_idx)}")

In [ ]:
subset_number = 5  # Change this to 2, 3, or 4 for the other DREAM4 networks

# 1. Load your multifactorial data
multifactorial_df = pd.read_csv(f"../experiments/dream4/insilico_size100_{subset_number}_multifactorial.tsv", sep='\t')
NODE_ORDER = list(multifactorial_df.columns)

# 2. Load the DREAM4 Gold Standard and give it headers
dream4_gs = pd.read_csv(f"../experiments/dream4/DREAM4_GoldStandard_InSilico_Size100_multifactorial_{subset_number}.tsv", 
                        sep='\t', header=None, names=['cause', 'outcome', 'edge'])

# 3. Generate the true binary DAG
binary_dag = sanitize_to_dag(get_dream4_ground_truth_dag(dream4_gs, NODE_ORDER))

# 4. Fit the Oracle Weights to get your linear SEM
dream4_data_array = multifactorial_df.to_numpy()
oracle_dag = fit_oracle_weights(binary_dag, dream4_data_array)

forward_pairs_idx = []
for node in binary_dag.nodes():
    # nx.descendants finds all nodes downstream (direct and indirect)
    for descendant in nx.descendants(binary_dag, node):
        forward_pairs_idx.append([node, descendant])

print(f"Number of forward pairs in DREAM4 subset {subset_number}: {len(forward_pairs_idx)}")

In [ ]:


def run_experiment_dream4(subset_number, sampling_strategy="local", frac_req=0.3):

    # 1. Load your multifactorial data
    multifactorial_df = pd.read_csv(f"../experiments/dream4/insilico_size100_{subset_number}_multifactorial.tsv", sep='\t')
    NODE_ORDER = list(multifactorial_df.columns)

    # 2. Load the DREAM4 Gold Standard and give it headers
    dream4_gs = pd.read_csv(f"../experiments/dream4/DREAM4_GoldStandard_InSilico_Size100_multifactorial_{subset_number}.tsv", 
                            sep='\t', header=None, names=['cause', 'outcome', 'edge'])

    # 3. Generate the true binary DAG
    binary_dag = sanitize_to_dag(get_dream4_ground_truth_dag(dream4_gs, NODE_ORDER))

    # 4. Fit the Oracle Weights to get your linear SEM
    dream4_data_array = multifactorial_df.to_numpy()
    oracle_dag = fit_oracle_weights(binary_dag, dream4_data_array)

    forward_pairs_idx = []
    for node in binary_dag.nodes():
        # nx.descendants finds all nodes downstream (direct and indirect)
        for descendant in nx.descendants(binary_dag, node):
            forward_pairs_idx.append([node, descendant])


    base_args_skeleton = {
        "data_name": f"dream4_net{subset_number}",
        "real_data": dream4_data_array,
        "true_dag": oracle_dag,
        "observed": dream4_data_array.shape[1],
        "alpha": 0.01,
        "ci_test": "fisherz",
        "sampling_strategy": sampling_strategy
    }

    base_seed = 42
    frac_forb = 0

    base_args_skeleton["fraction_forbidden"] = frac_forb
    base_args_skeleton["fraction_required"] = frac_req

    # Run experiment
    result_dict_dream4, details_dict_dream4 = run_experiment_mpdag_real(base_args_skeleton, base_seed, forward_pairs_idx)
    return result_dict_dream4, details_dict_dream4

In [ ]:
import os
import csv
import numpy as np

output_file = "../experiments/dream4/b-load_results.csv"
os.makedirs(os.path.dirname(output_file), exist_ok=True)

fieldnames = [
    "subset_number", "sampling_strategy", "frac_req",
    # b-LOAD
    "precision_with_b_mean", "precision_with_b_std",
    "recall_with_b_mean",    "recall_with_b_std",
    "f1_with_b_mean",        "f1_with_b_std",
    "int_dist_with_b_mean",  "int_dist_with_b_std",
    # LOAD
    "precision_without_b_mean", "precision_without_b_std",
    "recall_without_b_mean",    "recall_without_b_std",
    "f1_without_b_mean",        "f1_without_b_std",
    "int_dist_without_b_mean",  "int_dist_without_b_std",
    # PC
    "precision_pc_mean", "precision_pc_std",
    "recall_pc_mean",    "recall_pc_std",
    "f1_pc_mean",        "f1_pc_std",
    "int_dist_pc_mean",  "int_dist_pc_std",
    # b-PC
    "precision_bpc_mean", "precision_bpc_std",
    "recall_bpc_mean",    "recall_bpc_std",
    "f1_bpc_mean",        "f1_bpc_std",
    "int_dist_bpc_mean",  "int_dist_bpc_std",
]

with open(output_file, mode='w', newline='') as f:
    csv.DictWriter(f, fieldnames=fieldnames).writeheader()

for subset_number in tqdm(range(1, 6), desc="DREAM4 Subsets"):
    for sampling_strategy in tqdm(["local", "global"], desc=f"Strategy (Subset {subset_number})", leave=False):
        for frac_req in tqdm([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
                              desc=f"Frac Req ({sampling_strategy})", leave=False):

            result_dict, _ = run_experiment_dream4(subset_number, sampling_strategy, frac_req)

            row_data = {"subset_number": subset_number,
                        "sampling_strategy": sampling_strategy,
                        "frac_req": frac_req}

            # Flatten result_dict: key format is e.g. "precision_with_b" -> mean/std
            for metric, stats in result_dict.items():
                row_data[f"{metric}_mean"] = float(stats["mean"])
                row_data[f"{metric}_std"]  = float(stats["std"])

            with open(output_file, mode='a', newline='') as f:
                csv.DictWriter(f, fieldnames=fieldnames).writerow(row_data)

print(f"Done. Results saved to {output_file}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

# ── Config ────────────────────────────────────────────────────────────────────
CSV_PATH   = "../experiments/dream4/b-load_results.csv"
OUTPUT_DIR = "../experiments/dream4/plots_0517"
os.makedirs(OUTPUT_DIR, exist_ok=True)

METRICS = ["precision", "recall", "f1", "int_dist"]
METRIC_LABELS = {"precision": "Precision", "recall": "Recall",
                 "f1": "F1 Score", "int_dist": "Int. Distance"}

# Four algorithms, four colour families
METHODS = [
    ("with_b",    "b-LOAD", "#1f77b4"),   # blue
    ("without_b", "LOAD",   "#aec7e8"),   # light blue
    ("bpc",       "b-PC",   "#d62728"),   # red
    ("pc",        "PC",     "#f4a582"),   # light red/salmon
]

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH, sep=None, engine="python")
df.columns = df.columns.str.strip()

subsets    = sorted(df["subset_number"].unique())
strategies = sorted(df["sampling_strategy"].unique())
fracs      = sorted(df["frac_req"].unique())
n_fracs    = len(fracs)

# bar layout: 4 method-groups per frac_req tick, small gap between groups
BAR_W  = 0.18
OFFSETS = np.array([-1.5, -0.5, 0.5, 1.5]) * BAR_W

# ── Plot ──────────────────────────────────────────────────────────────────────
for subset in subsets:
    for strategy in strategies:
        sub = df[(df["subset_number"] == subset) & (df["sampling_strategy"] == strategy)
                 ].sort_values("frac_req")
        if sub.empty:
            continue

        fig, axes = plt.subplots(1, len(METRICS), figsize=(6 * len(METRICS), 5), sharey=False)
        fig.suptitle(f"Subset {subset}  ·  Strategy: {strategy}",
                     fontsize=14, fontweight="bold", y=1.01)

        x = np.arange(n_fracs)

        for ax, metric in zip(axes, METRICS):
            for (suffix, label, color), offset in zip(METHODS, OFFSETS):
                col_mean = f"{metric}_{suffix}_mean"
                if col_mean not in sub.columns:
                    continue
                vals = [sub[sub["frac_req"] == f][col_mean].values[0]
                        if not sub[sub["frac_req"] == f].empty else 0.0
                        for f in fracs]
                ax.bar(x + offset, vals, width=BAR_W, color=color, label=label)

            ax.set_title(METRIC_LABELS[metric], fontsize=11)
            ax.set_xlabel("fraction required")
            ax.set_xticks(x)
            ax.set_xticklabels([f"{f:.1f}" for f in fracs], rotation=45, fontsize=8)
            ax.set_ylim(0, None)
            ax.grid(axis="y", alpha=0.3)

        # Shared legend
        handles = [mpatches.Patch(color=c, label=lbl) for _, lbl, c in METHODS]
        fig.legend(handles=handles, loc="upper center",
                   bbox_to_anchor=(0.5, -0.04), ncol=4, fontsize=9,
                   title="b-LOAD / LOAD / b-PC / PC", title_fontsize=9)

        plt.tight_layout()
        fname = f"{OUTPUT_DIR}/subset{subset}_{strategy}.png"
        plt.savefig(fname, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved: {fname}")

print("DREAM4 plotting done.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

CSV_PATH = "../experiments/dream4/b-load_results.csv"

# 1. Keep only F1 and Int. Distance
METRICS = ["f1", "int_dist"]
METRIC_LABELS = {"f1": "F1 Score", "int_dist": "Int. Distance"}

METHODS = [
    ("with_b",    "b-LOAD", "#1f77b4"),
    ("without_b", "LOAD",   "#aec7e8"),
    ("bpc",       "b-PC",   "#d62728"),
    ("pc",        "PC",     "#f4a582"),
]

# Load and clean data
df = pd.read_csv(CSV_PATH, sep=None, engine="python")
df.columns = df.columns.str.strip()

# 2. Aggregate results across the subsets (mean)
# Safely identify the relevant metric columns to average
metric_cols = [f"{m}_{s}_mean" for m in METRICS for s, _, _ in METHODS]
metric_cols = [c for c in metric_cols if c in df.columns]

# Group by strategy and frac_req, taking the mean across the subsets
df_agg = df.groupby(["sampling_strategy", "frac_req"])[metric_cols].mean().reset_index()

strategies = sorted(df_agg["sampling_strategy"].unique())
fracs      = sorted(df_agg["frac_req"].unique())
x          = np.arange(len(fracs))

BAR_W   = 0.18
OFFSETS = np.array([-1.5, -0.5, 0.5, 1.5]) * BAR_W

# 3. Create a single grid (Rows = Strategies, Columns = Metrics). No title as requested.
n_rows, n_cols = len(strategies), len(METRICS)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows), squeeze=False, sharex=True)

for i, strategy in enumerate(strategies):
    # Set the index to frac_req once per strategy for clean alignment
    sub = df_agg[df_agg["sampling_strategy"] == strategy].set_index("frac_req")
    
    for j, metric in enumerate(METRICS):
        ax = axes[i, j]
        if sub.empty: continue

        # Plot bars
        for (suffix, label, color), offset in zip(METHODS, OFFSETS):
            col_mean = f"{metric}_{suffix}_mean"
            if col_mean in sub.columns:
                # reindex ensures all fracs are aligned, missing fracs get 0.0
                vals = sub[col_mean].reindex(fracs, fill_value=0.0).values
                ax.bar(x + offset, vals, width=BAR_W, color=color, label=label)

        # 4. Handle labeling cleanly to avoid redundancy
        if i == 0: # Top row: Metric titles
            ax.set_title(METRIC_LABELS[metric], fontsize=13, fontweight="bold")
        if j == 0: # Left column: Strategy names
            ax.set_ylabel(f"Strategy: {strategy}\n\nScore", fontsize=11, fontweight="bold")
        
        if i == n_rows - 1: # Bottom row: X-axis labels
            ax.set_xlabel("Fraction Required", fontsize=11)
            ax.set_xticks(x)
            ax.set_xticklabels([f"{f:.1f}" for f in fracs], rotation=45, fontsize=9)

        ax.set_ylim(0, None)
        ax.grid(axis="y", alpha=0.3)

# 5. Create a single, shared legend at the bottom of the figure
handles = [mpatches.Patch(color=c, label=lbl) for _, lbl, c in METHODS]
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0), 
           ncol=4, fontsize=10, title="Methods", title_fontsize=11)

plt.tight_layout()

# 6. Display the plot directly
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import scienceplots

# Apply SciencePlots style
plt.style.use(["science", "no-latex"])

CSV_PATH = "../experiments/dream4/b-load_results.csv"

# 1. Keep only F1 and Int. Distance
METRICS = ["f1", "int_dist"]
METRIC_LABELS = {"f1": "F1 Score", "int_dist": "Int. Distance"}

METHODS = [
    ("with_b",    "b-LOAD", "#1f77b4"),
    ("without_b", "LOAD",   "#aec7e8"),
    ("bpc",       "b-PC",   "#d62728"),
    ("pc",        "PC",     "#f4a582"),
]

# Load and clean data
df = pd.read_csv(CSV_PATH, sep=None, engine="python")
df.columns = df.columns.str.strip()

# 2. Aggregate results across the subsets (mean)
# Safely identify the relevant metric columns to average
metric_cols = [f"{m}_{s}_mean" for m in METRICS for s, _, _ in METHODS]
metric_cols = [c for c in metric_cols if c in df.columns]

# Group by strategy and frac_req, taking the mean across the subsets
df_agg = df.groupby(["sampling_strategy", "frac_req"])[metric_cols].mean().reset_index()

strategies = sorted(df_agg["sampling_strategy"].unique())
fracs      = sorted(df_agg["frac_req"].unique())
x          = np.arange(len(fracs))

BAR_W   = 0.18
OFFSETS = np.array([-1.5, -0.5, 0.5, 1.5]) * BAR_W

# 3. Increased figsize to support the massive fonts
n_rows, n_cols = len(strategies), len(METRICS)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 6 * n_rows), squeeze=False, sharex=True)

for i, strategy in enumerate(strategies):
    # Set the index to frac_req once per strategy for clean alignment
    sub = df_agg[df_agg["sampling_strategy"] == strategy].set_index("frac_req")
    
    for j, metric in enumerate(METRICS):
        ax = axes[i, j]
        if sub.empty: continue

        # Plot bars
        for (suffix, label, color), offset in zip(METHODS, OFFSETS):
            col_mean = f"{metric}_{suffix}_mean"
            if col_mean in sub.columns:
                # reindex ensures all fracs are aligned, missing fracs get 0.0
                vals = sub[col_mean].reindex(fracs, fill_value=0.0).values
                ax.bar(x + offset, vals, width=BAR_W, color=color, label=label)

        # 4. Handle massive labeling cleanly
        if i == 0: # Top row: Metric titles
            ax.set_title(METRIC_LABELS[metric], fontsize=26, fontweight="bold", pad=15)
            
        if j == 0: # Left column: Strategy names
            ax.set_ylabel(f"Strategy: {strategy}\n\nScore", fontsize=24, fontweight="bold", labelpad=12)
        
        if i == n_rows - 1: # Bottom row: X-axis labels
            ax.set_xlabel("Fraction Required", fontsize=24, fontweight="bold", labelpad=10)
            ax.set_xticks(x)
            # Increased tick label size and kept rotation
            ax.set_xticklabels([f"{f:.1f}" for f in fracs], rotation=45, fontsize=18)

        # Tick parameters for Y-axis
        ax.tick_params(axis='y', which='major', labelsize=18)
        ax.set_ylim(0, None)
        ax.grid(axis="y", alpha=0.3, linestyle='--')

# 5. Apply tight_layout BEFORE adding the external legend to lock the grid in place
plt.tight_layout(w_pad=2.0, h_pad=2.0)

# Force a margin at the top of the entire figure
fig.subplots_adjust(top=0.88)

# 6. Create a single, shared legend at the top of the figure
handles = [mpatches.Patch(color=c, label=lbl) for _, lbl, c in METHODS]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.90), 
           ncol=4, frameon=False, prop={'size': 24, 'weight': 'bold'}, columnspacing=2.0)

# Save the plot directly to PDF
plt.savefig("dream4_results_bar.pdf", bbox_inches="tight")
plt.show()

## Exp 4: application to DREAM5 dataset

In [ ]:
goldstandard_path = "experiments/dream5/net2/dream5_net2_goldstandard.tsv"
dream5_net2_gs = pd.read_csv(
    goldstandard_path,
    sep='\t',
    header=None,
    names=['cause', 'outcome', 'direction']
)

dream5_net2_gs.head()

In [ ]:
goldstandard_path = "experiments/dream5/net2/dream5_net2_goldstandard.tsv"
dream5_net2_gs = pd.read_csv(
    goldstandard_path,
    sep='\t',
    header=None,
    names=['cause', 'outcome', 'direction']
)


# Filter rows where both 'cause' and 'outcome' are within G1-G99
def is_valid_gene(g):
    if g.startswith('G'):
        try:
            num = int(g[1:])
            return 1 <= num <= 99
        except ValueError:
            return False
    return False

dream5_net2_gs = dream5_net2_gs[
    dream5_net2_gs['cause'].apply(is_valid_gene) &
    dream5_net2_gs['outcome'].apply(is_valid_gene)
]

In [ ]:
tf_path = "../experiments/dream5/net2/net2_transcription_factors.tsv"
tf_net2 = pd.read_csv(
    tf_path,
    sep='\t',
    header=None,
    names=['gene_name']
)



In [ ]:
TF_ORDER = tf_net2.values.flatten().tolist()
TF_ORDER

In [ ]:
# Filter rows where both 'cause' and 'outcome' are within G1-G99
def is_valid_gene(g):
    if g.startswith('G'):
        try:
            num = int(g[1:])
            return 1 <= num <= 99
        except ValueError:
            return False
    return False

dream5_net2_gs = dream5_net2_gs[
    dream5_net2_gs['cause'].apply(is_valid_gene) &
    dream5_net2_gs['outcome'].apply(is_valid_gene)
]

In [ ]:
dream5_net2_gs

In [ ]:
import networkx as nx
import pandas as pd

# You MUST define this list so it exactly matches the column order of your 
# filtered 99-column expression dataset! 
# (e.g., TF_ORDER = ['G1', 'G2', 'G3', ..., 'G99'])
# You can usually get this from: tf_order = list(filtered_expression_df.columns)

def get_dream5_tf_ground_truth_dag(gs_dataframe: pd.DataFrame, tf_order: list) -> nx.DiGraph:
    """
    Constructs the ground truth DAG for the DREAM5 dataset, restricted to a specific list of TFs.
    Nodes are integer indices corresponding to the order in tf_order.
    """
    # 1. Create the reverse mapping (name -> index)
    name_to_idx = {name: idx for idx, name in enumerate(tf_order)}

    true_dag = nx.DiGraph()

    # 2. Add ALL nodes as integer indices (0 through 98)
    # CRITICAL: This ensures disconnected TFs still exist in the math!
    true_dag.add_nodes_from(range(len(tf_order)))

    # 3. Extract edges and map string names to integer indices
    edges_by_index = []
    for _, row in gs_dataframe.iterrows():
        cause_name = row['cause']
        outcome_name = row['outcome']
        
        # Safety check: Only add the edge if BOTH nodes are in our 99 TF list
        if cause_name in name_to_idx and outcome_name in name_to_idx:
            u = name_to_idx[cause_name]
            v = name_to_idx[outcome_name]
            edges_by_index.append((u, v))

    # 4. Add the integer edges to the graph
    true_dag.add_edges_from(edges_by_index)

    return true_dag

# --- How to use it in your pipeline ---
# Assuming your DataFrame from the screenshot is called `dream5_net2_gs`
binary_dag = get_dream5_tf_ground_truth_dag(dream5_net2_gs, TF_ORDER)

# Now it is ready for the Oracle!
# oracle_dag = fit_oracle_weights(binary_dag, filtered_expression_data)

In [ ]:
draw_ground_truth_from_dag(binary_dag)

In [ ]:
tf_path = "../experiments/dream5/net2/net2_transcription_factors.tsv"
tf_net2 = pd.read_csv(
    tf_path,
    sep='\t',
    header=None,
    names=['gene_name']
)

TF_ORDER = tf_net2.values.flatten().tolist()


data_path = "../experiments/dream5/net2/net2_expression_data.tsv"
dream5_net2_data = pd.read_csv(
    data_path,
    sep='\t',
)

filtered_expression_data = dream5_net2_data[TF_ORDER]

import networkx as nx
import pandas as pd

# You MUST define this list so it exactly matches the column order of your 
# filtered 99-column expression dataset! 
# (e.g., TF_ORDER = ['G1', 'G2', 'G3', ..., 'G99'])
# You can usually get this from: tf_order = list(filtered_expression_df.columns)

def get_dream5_tf_ground_truth_dag(gs_dataframe: pd.DataFrame, tf_order: list) -> nx.DiGraph:
    """
    Constructs the ground truth DAG for the DREAM5 dataset, restricted to a specific list of TFs.
    Nodes are integer indices corresponding to the order in tf_order.
    """
    # 1. Create the reverse mapping (name -> index)
    name_to_idx = {name: idx for idx, name in enumerate(tf_order)}

    true_dag = nx.DiGraph()

    # 2. Add ALL nodes as integer indices (0 through 98)
    # CRITICAL: This ensures disconnected TFs still exist in the math!
    true_dag.add_nodes_from(range(len(tf_order)))

    # 3. Extract edges and map string names to integer indices
    edges_by_index = []
    for _, row in gs_dataframe.iterrows():
        cause_name = row['cause']
        outcome_name = row['outcome']
        
        # Safety check: Only add the edge if BOTH nodes are in our 99 TF list
        if cause_name in name_to_idx and outcome_name in name_to_idx:
            u = name_to_idx[cause_name]
            v = name_to_idx[outcome_name]
            edges_by_index.append((u, v))

    # 4. Add the integer edges to the graph
    true_dag.add_edges_from(edges_by_index)

    return true_dag

# --- How to use it in your pipeline ---
# Assuming your DataFrame from the screenshot is called `dream5_net2_gs`
binary_dag = get_dream5_tf_ground_truth_dag(dream5_net2_gs, TF_ORDER)

# Now it is ready for the Oracle!
# oracle_dag = fit_oracle_weights(binary_dag, filtered_expression_data)

# Now it is ready for the Oracle!
oracle_dag_dream5 = fit_oracle_weights(binary_dag, filtered_expression_data.to_numpy())

In [ ]:
filtered_expression_data = dream5_net2_data[TF_ORDER]
filtered_expression_data.head()

In [ ]:
filtered_expression_data.shape

In [ ]:
# Now it is ready for the Oracle!
oracle_dag_dream5 = fit_oracle_weights(binary_dag, filtered_expression_data.to_numpy())

In [ ]:
import networkx as nx

# 1. Get all mathematically identifiable forward pairs (Ancestor -> Descendant)
# This uses the true_dag you generated earlier
forward_pairs_idx = []
for node in binary_dag.nodes():
    # nx.descendants finds all nodes downstream (direct and indirect)
    for descendant in nx.descendants(binary_dag, node):
        forward_pairs_idx.append([node, descendant])

In [ ]:
forward_pairs_idx

In [ ]:
from src.real_data_applications.sachs import fit_oracle_weights



base_args_skeleton = {
      "data_name": "dream5_net2",
      "real_data": filtered_expression_data.to_numpy(),
      "true_dag": oracle_dag_dream5,
      "observed": filtered_expression_data.to_numpy().shape[1],
      "alpha": 0.01,
      "ci_test": "fisherz",
      "sampling_strategy": "local"
}

base_seed = 42
frac_forb = 0
frac_req = 0.3

base_args_skeleton["fraction_forbidden"] = frac_forb
base_args_skeleton["fraction_required"] = frac_req

# Run experiment
result_dict_dream5, details_dict_dream5 = run_experiment_mpdag_real(base_args_skeleton, base_seed, forward_pairs_idx)

In [ ]:
result_dict_dream5

In [ ]:


def test_dream5(sampling_strategy, frac_req):
    base_args_skeleton = {
        "data_name": "dream5_net2",
        "real_data": filtered_expression_data.to_numpy(),
        "true_dag": oracle_dag_dream5,
        "observed": filtered_expression_data.to_numpy().shape[1],
        "alpha": 0.01,
        "ci_test": "fisherz",
        "sampling_strategy": sampling_strategy
    }

    base_seed = 42
    frac_forb = 0
    frac_req = frac_req

    base_args_skeleton["fraction_forbidden"] = frac_forb
    base_args_skeleton["fraction_required"] = frac_req

    # Run experiment
    result_dict_dream5, details_dict_dream5 = run_experiment_mpdag_real(base_args_skeleton, base_seed, forward_pairs_idx)
    return result_dict_dream5, details_dict_dream5

for sampling_strategy in tqdm(["local", "global"]):
    for frac_req in tqdm([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]):
        print(f"Testing sampling_strategy={sampling_strategy}, frac_req={frac_req}")
        result_dict, details_dict = test_dream5(sampling_strategy, frac_req)
        print(f"Results: {result_dict}\n")

### Full sweep — all strategies × frac_req values, save to CSV

In [ ]:
tf_path = "../experiments/dream5/net2/net2_transcription_factors.tsv"
tf_net2 = pd.read_csv(
    tf_path,
    sep='\t',
    header=None,
    names=['gene_name']
)

TF_ORDER = tf_net2.values.flatten().tolist()


data_path = "../experiments/dream5/net2/net2_expression_data.tsv"
dream5_net2_data = pd.read_csv(
    data_path,
    sep='\t',
)

filtered_expression_data = dream5_net2_data[TF_ORDER]

import networkx as nx
import pandas as pd

# You MUST define this list so it exactly matches the column order of your 
# filtered 99-column expression dataset! 
# (e.g., TF_ORDER = ['G1', 'G2', 'G3', ..., 'G99'])
# You can usually get this from: tf_order = list(filtered_expression_df.columns)

def get_dream5_tf_ground_truth_dag(gs_dataframe: pd.DataFrame, tf_order: list) -> nx.DiGraph:
    """
    Constructs the ground truth DAG for the DREAM5 dataset, restricted to a specific list of TFs.
    Nodes are integer indices corresponding to the order in tf_order.
    """
    # 1. Create the reverse mapping (name -> index)
    name_to_idx = {name: idx for idx, name in enumerate(tf_order)}

    true_dag = nx.DiGraph()

    # 2. Add ALL nodes as integer indices (0 through 98)
    # CRITICAL: This ensures disconnected TFs still exist in the math!
    true_dag.add_nodes_from(range(len(tf_order)))

    # 3. Extract edges and map string names to integer indices
    edges_by_index = []
    for _, row in gs_dataframe.iterrows():
        cause_name = row['cause']
        outcome_name = row['outcome']
        
        # Safety check: Only add the edge if BOTH nodes are in our 99 TF list
        if cause_name in name_to_idx and outcome_name in name_to_idx:
            u = name_to_idx[cause_name]
            v = name_to_idx[outcome_name]
            edges_by_index.append((u, v))

    # 4. Add the integer edges to the graph
    true_dag.add_edges_from(edges_by_index)

    return true_dag


goldstandard_path = "../experiments/dream5/net2/dream5_net2_goldstandard.tsv"
dream5_net2_gs = pd.read_csv(
    goldstandard_path,
    sep='\t',
    header=None,
    names=['cause', 'outcome', 'direction']
)


# Filter rows where both 'cause' and 'outcome' are within G1-G99
def is_valid_gene(g):
    if g.startswith('G'):
        try:
            num = int(g[1:])
            return 1 <= num <= 99
        except ValueError:
            return False
    return False

dream5_net2_gs = dream5_net2_gs[
    dream5_net2_gs['cause'].apply(is_valid_gene) &
    dream5_net2_gs['outcome'].apply(is_valid_gene)
]

# --- How to use it in your pipeline ---
# Assuming your DataFrame from the screenshot is called `dream5_net2_gs`
binary_dag = get_dream5_tf_ground_truth_dag(dream5_net2_gs, TF_ORDER)

import networkx as nx

# 1. Get all mathematically identifiable forward pairs (Ancestor -> Descendant)
# This uses the true_dag you generated earlier
forward_pairs_idx = []
for node in binary_dag.nodes():
    # nx.descendants finds all nodes downstream (direct and indirect)
    for descendant in nx.descendants(binary_dag, node):
        forward_pairs_idx.append([node, descendant])

# Now it is ready for the Oracle!
# oracle_dag = fit_oracle_weights(binary_dag, filtered_expression_data)

# Now it is ready for the Oracle!
oracle_dag_dream5 = fit_oracle_weights(binary_dag, filtered_expression_data.to_numpy())

In [ ]:
import os
import csv

output_file_dream5 = "../experiments/dream5/b-load_results.csv"
os.makedirs(os.path.dirname(output_file_dream5), exist_ok=True)

fieldnames = [
    "dataset", "sampling_strategy", "frac_req",
    # b-LOAD
    "precision_with_b_mean", "precision_with_b_std",
    "recall_with_b_mean",    "recall_with_b_std",
    "f1_with_b_mean",        "f1_with_b_std",
    "int_dist_with_b_mean",  "int_dist_with_b_std",
    # LOAD
    "precision_without_b_mean", "precision_without_b_std",
    "recall_without_b_mean",    "recall_without_b_std",
    "f1_without_b_mean",        "f1_without_b_std",
    "int_dist_without_b_mean",  "int_dist_without_b_std",
    # PC
    "precision_pc_mean", "precision_pc_std",
    "recall_pc_mean",    "recall_pc_std",
    "f1_pc_mean",        "f1_pc_std",
    "int_dist_pc_mean",  "int_dist_pc_std",
    # b-PC
    "precision_bpc_mean", "precision_bpc_std",
    "recall_bpc_mean",    "recall_bpc_std",
    "f1_bpc_mean",        "f1_bpc_std",
    "int_dist_bpc_mean",  "int_dist_bpc_std",
]

with open(output_file_dream5, mode='w', newline='') as f:
    csv.DictWriter(f, fieldnames=fieldnames).writeheader()

base_args_dream5 = {
    "data_name":  "dream5_net2",
    "real_data":  filtered_expression_data.to_numpy(),
    "true_dag":   oracle_dag_dream5,
    "observed":   filtered_expression_data.shape[1],
    "alpha":      0.01,
    "ci_test":    "fisherz",
    "fraction_forbidden": 0.0,
}

for sampling_strategy in tqdm(["local", "global"], desc="Strategy"):
    for frac_req in tqdm([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
                         desc=f"Frac Req ({sampling_strategy})", leave=False):
        base_args_dream5["sampling_strategy"]  = sampling_strategy
        base_args_dream5["fraction_required"]  = frac_req

        result_dict, _ = run_experiment_mpdag_real(base_args_dream5, 42, forward_pairs_idx)

        row_data = {"dataset": "dream5_net2",
                    "sampling_strategy": sampling_strategy,
                    "frac_req": frac_req}
        for metric, stats in result_dict.items():
            row_data[f"{metric}_mean"] = float(stats["mean"])
            row_data[f"{metric}_std"]  = float(stats["std"])

        with open(output_file_dream5, mode='a', newline='') as f:
            csv.DictWriter(f, fieldnames=fieldnames).writerow(row_data)

print(f"Done. Results saved to {output_file_dream5}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

CSV_PATH_D5 = "../experiments/dream5/b-load_results.csv"
OUTPUT_DIR_D5 = "../experiments/dream5/plots"
os.makedirs(OUTPUT_DIR_D5, exist_ok=True)

METRICS = ["precision", "recall", "f1", "int_dist"]
METRIC_LABELS = {"precision": "Precision", "recall": "Recall",
                 "f1": "F1 Score", "int_dist": "Int. Distance"}

METHODS = [
    ("with_b",    "b-LOAD", "#1f77b4"),
    ("without_b", "LOAD",   "#aec7e8"),
    ("bpc",       "b-PC",   "#d62728"),
    ("pc",        "PC",     "#f4a582"),
]

df_d5 = pd.read_csv(CSV_PATH_D5, sep=None, engine="python")
df_d5.columns = df_d5.columns.str.strip()

strategies = sorted(df_d5["sampling_strategy"].unique())
fracs      = sorted(df_d5["frac_req"].unique())
n_fracs    = len(fracs)
BAR_W   = 0.18
OFFSETS = np.array([-1.5, -0.5, 0.5, 1.5]) * BAR_W
x = np.arange(n_fracs)

for strategy in strategies:
    sub = df_d5[df_d5["sampling_strategy"] == strategy].sort_values("frac_req")
    if sub.empty:
        continue

    fig, axes = plt.subplots(1, len(METRICS), figsize=(6 * len(METRICS), 5))
    fig.suptitle(f"DREAM5 net2  ·  Strategy: {strategy}",
                 fontsize=14, fontweight="bold", y=1.01)

    for ax, metric in zip(axes, METRICS):
        for (suffix, label, color), offset in zip(METHODS, OFFSETS):
            col_mean = f"{metric}_{suffix}_mean"
            if col_mean not in sub.columns:
                continue
            vals = [sub[sub["frac_req"] == f][col_mean].values[0]
                    if not sub[sub["frac_req"] == f].empty else 0.0
                    for f in fracs]
            ax.bar(x + offset, vals, width=BAR_W, color=color, label=label)

        ax.set_title(METRIC_LABELS[metric], fontsize=11)
        ax.set_xlabel("fraction required")
        ax.set_xticks(x)
        ax.set_xticklabels([f"{f:.1f}" for f in fracs], rotation=45, fontsize=8)
        ax.set_ylim(0, None)
        ax.grid(axis="y", alpha=0.3)

    handles = [mpatches.Patch(color=c, label=lbl) for _, lbl, c in METHODS]
    fig.legend(handles=handles, loc="upper center",
               bbox_to_anchor=(0.5, -0.04), ncol=4, fontsize=9,
               title="b-LOAD / LOAD / b-PC / PC", title_fontsize=9)
    plt.tight_layout()
    fname = f"{OUTPUT_DIR_D5}/dream5_net2_{strategy}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {fname}")

print("DREAM5 plotting done.")

## Exp 2: application to Sachs data — full sweep (all strategies × frac_req)

In [ ]:
import pandas as pd

# observational only
obs = pd.read_csv(
    "https://www.bnlearn.com/book-crc/code/sachs.data.txt.gz",
    sep="\t"
)
# full interventional (INT column tells you which protein was perturbed, 0 = observational)
full = pd.read_csv("https://www.bnlearn.com/book-crc/code/sachs.interventional.txt.gz", sep=" ")
obs_only = full[full["INT"] == 0]

In [ ]:
obs

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

In [ ]:
sachs_data

In [ ]:
import numpy as np
obs.to_numpy()

In [ ]:
from src.real_data_applications.sachs import (
    fit_oracle_weights, get_protein_name, get_sachs_ground_truth_dag, forward_pairs_idx
)
from causallearn.utils.Dataset import load_dataset

true_dag_sachs_bin = get_sachs_ground_truth_dag()
sachs_data, sachs_labels = load_dataset("sachs")

#### Replacing sachs_data with obs_only (observational samples only)
sachs_data = obs.to_numpy()

oracle_dag_sachs = fit_oracle_weights(true_dag_sachs_bin, sachs_data)

print(f"Sachs: {sachs_data.shape[0]} samples, {sachs_data.shape[1]} nodes, "
      f"{len(forward_pairs_idx)} forward pairs")

In [ ]:
oracle_dag_sachs

In [ ]:
import os, csv

output_file_sachs = "experiments/sachs/b-load_results.csv"
os.makedirs(os.path.dirname(output_file_sachs), exist_ok=True)

fieldnames = [
    "dataset", "sampling_strategy", "frac_req",
    # b-LOAD
    "precision_with_b_mean", "precision_with_b_std",
    "recall_with_b_mean",    "recall_with_b_std",
    "f1_with_b_mean",        "f1_with_b_std",
    "int_dist_with_b_mean",  "int_dist_with_b_std",
    # LOAD
    "precision_without_b_mean", "precision_without_b_std",
    "recall_without_b_mean",    "recall_without_b_std",
    "f1_without_b_mean",        "f1_without_b_std",
    "int_dist_without_b_mean",  "int_dist_without_b_std",
    # PC
    "precision_pc_mean", "precision_pc_std",
    "recall_pc_mean",    "recall_pc_std",
    "f1_pc_mean",        "f1_pc_std",
    "int_dist_pc_mean",  "int_dist_pc_std",
    # b-PC
    "precision_bpc_mean", "precision_bpc_std",
    "recall_bpc_mean",    "recall_bpc_std",
    "f1_bpc_mean",        "f1_bpc_std",
    "int_dist_bpc_mean",  "int_dist_bpc_std",
]

with open(output_file_sachs, mode='w', newline='') as f:
    csv.DictWriter(f, fieldnames=fieldnames).writeheader()

base_args_sachs = {
    "data_name":  "sachs",
    "real_data":  sachs_data,
    "true_dag":   oracle_dag_sachs,
    "observed":   sachs_data.shape[1],
    "alpha":      0.01,
    "ci_test":    "fisherz",
    "fraction_forbidden": 0.0,
}

for sampling_strategy in tqdm(["local", "global"], desc="Strategy"):
    for frac_req in tqdm([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
                         desc=f"Frac Req ({sampling_strategy})", leave=False):
        base_args_sachs["sampling_strategy"] = sampling_strategy
        base_args_sachs["fraction_required"] = frac_req

        result_dict, _ = run_experiment_mpdag_real(base_args_sachs, 42, forward_pairs_idx)

        row_data = {"dataset": "sachs",
                    "sampling_strategy": sampling_strategy,
                    "frac_req": frac_req}
        for metric, stats in result_dict.items():
            row_data[f"{metric}_mean"] = float(stats["mean"])
            row_data[f"{metric}_std"]  = float(stats["std"])

        with open(output_file_sachs, mode='a', newline='') as f:
            csv.DictWriter(f, fieldnames=fieldnames).writerow(row_data)

print(f"Done. Results saved to {output_file_sachs}")

In [ ]:
import os, csv

output_file_sachs = "../experiments/sachs/b-load_results_new.csv"
os.makedirs(os.path.dirname(output_file_sachs), exist_ok=True)

fieldnames = [
    "dataset", "sampling_strategy", "frac_req",
    # b-LOAD
    "precision_with_b_mean", "precision_with_b_std",
    "recall_with_b_mean",    "recall_with_b_std",
    "f1_with_b_mean",        "f1_with_b_std",
    "int_dist_with_b_mean",  "int_dist_with_b_std",
    # LOAD
    "precision_without_b_mean", "precision_without_b_std",
    "recall_without_b_mean",    "recall_without_b_std",
    "f1_without_b_mean",        "f1_without_b_std",
    "int_dist_without_b_mean",  "int_dist_without_b_std",
    # PC
    "precision_pc_mean", "precision_pc_std",
    "recall_pc_mean",    "recall_pc_std",
    "f1_pc_mean",        "f1_pc_std",
    "int_dist_pc_mean",  "int_dist_pc_std",
    # b-PC
    "precision_bpc_mean", "precision_bpc_std",
    "recall_bpc_mean",    "recall_bpc_std",
    "f1_bpc_mean",        "f1_bpc_std",
    "int_dist_bpc_mean",  "int_dist_bpc_std",
]

with open(output_file_sachs, mode='w', newline='') as f:
    csv.DictWriter(f, fieldnames=fieldnames).writeheader()

base_args_sachs = {
    "data_name":  "sachs",
    "real_data":  sachs_data,
    "true_dag":   oracle_dag_sachs,
    "observed":   sachs_data.shape[1],
    "alpha":      0.01,
    "ci_test":    "fisherz",
    "fraction_forbidden": 0.0,
}

for sampling_strategy in tqdm(["local", "global"], desc="Strategy"):
    for frac_req in tqdm([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
                         desc=f"Frac Req ({sampling_strategy})", leave=False):
        base_args_sachs["sampling_strategy"] = sampling_strategy
        base_args_sachs["fraction_required"] = frac_req

        result_dict, _ = run_experiment_mpdag_real(base_args_sachs, 42, forward_pairs_idx)

        row_data = {"dataset": "sachs",
                    "sampling_strategy": sampling_strategy,
                    "frac_req": frac_req}
        for metric, stats in result_dict.items():
            row_data[f"{metric}_mean"] = float(stats["mean"])
            row_data[f"{metric}_std"]  = float(stats["std"])

        with open(output_file_sachs, mode='a', newline='') as f:
            csv.DictWriter(f, fieldnames=fieldnames).writerow(row_data)

print(f"Done. Results saved to {output_file_sachs}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

CSV_PATH_S = "../experiments/sachs/b-load_results.csv"
OUTPUT_DIR_S = "../experiments/sachs/plots"
os.makedirs(OUTPUT_DIR_S, exist_ok=True)

METRICS = ["precision", "recall", "f1", "int_dist"]
METRIC_LABELS = {"precision": "Precision", "recall": "Recall",
                 "f1": "F1 Score", "int_dist": "Int. Distance"}

METHODS = [
    ("with_b",    "b-LOAD", "#1f77b4"),
    ("without_b", "LOAD",   "#aec7e8"),
    ("bpc",       "b-PC",   "#d62728"),
    ("pc",        "PC",     "#f4a582"),
]

df_s = pd.read_csv(CSV_PATH_S, sep=None, engine="python")
df_s.columns = df_s.columns.str.strip()

strategies = sorted(df_s["sampling_strategy"].unique())
fracs      = sorted(df_s["frac_req"].unique())
n_fracs    = len(fracs)
BAR_W   = 0.18
OFFSETS = np.array([-1.5, -0.5, 0.5, 1.5]) * BAR_W
x = np.arange(n_fracs)

for strategy in strategies:
    sub = df_s[df_s["sampling_strategy"] == strategy].sort_values("frac_req")
    if sub.empty:
        continue

    fig, axes = plt.subplots(1, len(METRICS), figsize=(6 * len(METRICS), 5))
    fig.suptitle(f"Sachs  ·  Strategy: {strategy}",
                 fontsize=14, fontweight="bold", y=1.01)

    for ax, metric in zip(axes, METRICS):
        for (suffix, label, color), offset in zip(METHODS, OFFSETS):
            col_mean = f"{metric}_{suffix}_mean"
            if col_mean not in sub.columns:
                continue
            vals = [sub[sub["frac_req"] == f][col_mean].values[0]
                    if not sub[sub["frac_req"] == f].empty else 0.0
                    for f in fracs]
            ax.bar(x + offset, vals, width=BAR_W, color=color, label=label)

        ax.set_title(METRIC_LABELS[metric], fontsize=11)
        ax.set_xlabel("fraction required")
        ax.set_xticks(x)
        ax.set_xticklabels([f"{f:.1f}" for f in fracs], rotation=45, fontsize=8)
        ax.set_ylim(0, None)
        ax.grid(axis="y", alpha=0.3)

    handles = [mpatches.Patch(color=c, label=lbl) for _, lbl, c in METHODS]
    fig.legend(handles=handles, loc="upper center",
               bbox_to_anchor=(0.5, -0.04), ncol=4, fontsize=9,
               title="b-LOAD / LOAD / b-PC / PC", title_fontsize=9)
    plt.tight_layout()
    fname = f"{OUTPUT_DIR_S}/sachs_{strategy}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {fname}")

print("Sachs plotting done.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# import scienceplots
plt.style.use("science")

CSV_PATH_S = "../experiments/sachs/b-load_results.csv"

# 1. Keep only F1 and Int. Distance
METRICS = ["f1", "int_dist"]
METRIC_LABELS = {"f1": "F1 Score", "int_dist": "Int. Distance"}

METHODS = [
    ("with_b",    "b-LOAD", "#1f77b4"),
    ("without_b", "LOAD",   "#aec7e8"),
    ("bpc",       "b-PC",   "#d62728"),
    ("pc",        "PC",     "#f4a582"),
]

# Load and clean data
df_s = pd.read_csv(CSV_PATH_S, sep=None, engine="python")
df_s.columns = df_s.columns.str.strip()

strategies = sorted(df_s["sampling_strategy"].unique())
fracs      = sorted(df_s["frac_req"].unique())
x          = np.arange(len(fracs))

BAR_W   = 0.18
OFFSETS = np.array([-1.5, -0.5, 0.5, 1.5]) * BAR_W

# 2. Create a single grid (Rows = Strategies, Columns = Metrics)
n_rows, n_cols = len(strategies), len(METRICS)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows), squeeze=False, sharex=True)

for i, strategy in enumerate(strategies):
    # Set the index to frac_req once per strategy to eliminate nested list comprehension loops
    sub = df_s[df_s["sampling_strategy"] == strategy].set_index("frac_req")
    
    for j, metric in enumerate(METRICS):
        ax = axes[i, j]
        if sub.empty: continue

        # Plot bars
        for (suffix, label, color), offset in zip(METHODS, OFFSETS):
            col_mean = f"{metric}_{suffix}_mean"
            if col_mean in sub.columns:
                # reindex ensures all fracs are aligned, missing fracs get 0.0
                vals = sub[col_mean].reindex(fracs, fill_value=0.0).values
                ax.bar(x + offset, vals, width=BAR_W, color=color, label=label)

        # 3. Handle labeling cleanly to avoid redundancy
        if i == 0: # Top row: Metric titles
            ax.set_title(METRIC_LABELS[metric], fontsize=13, fontweight="bold")
        if j == 0: # Left column: Strategy names
            ax.set_ylabel(f"Strategy: {strategy}\n\nScore", fontsize=11, fontweight="bold")
        
        if i == n_rows - 1: # Bottom row: X-axis labels
            ax.set_xlabel("Fraction Required", fontsize=11)
            ax.set_xticks(x)
            ax.set_xticklabels([f"{f:.1f}" for f in fracs], rotation=45, fontsize=9)

        ax.set_ylim(0, None)
        ax.grid(axis="y", alpha=0.3)

# 4. Create a single, shared legend at the bottom of the figure
handles = [mpatches.Patch(color=c, label=lbl) for _, lbl, c in METHODS]
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0), 
           ncol=4, fontsize=10, title="Methods", title_fontsize=11)

plt.tight_layout()

# 5. Display the plot instead of saving it
plt.show()

In [ ]:
!pip install SciencePlots

In [ ]:
import matplotlib.pyplot as plt
import scienceplots

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Apply SciencePlots style
plt.style.use(["science", "no-latex"])

CSV_PATH_S = "../experiments/sachs/b-load_results_new.csv"

# 1. Keep only F1 and Int. Distance
METRICS = ["f1", "int_dist"]
METRIC_LABELS = {"f1": "F1 Score", "int_dist": "Int. Distance"}

METHODS = [
    ("with_b",    "b-LOAD", "#1f77b4"),
    ("without_b", "LOAD",   "#aec7e8"),
    ("bpc",       "b-PC",   "#d62728"),
    ("pc",        "PC",     "#f4a582"),
]

# Load and clean data
df_s = pd.read_csv(CSV_PATH_S, sep=None, engine="python")
df_s.columns = df_s.columns.str.strip()

strategies = sorted(df_s["sampling_strategy"].unique())
fracs      = sorted(df_s["frac_req"].unique())
x          = np.arange(len(fracs))

BAR_W   = 0.18
OFFSETS = np.array([-1.5, -0.5, 0.5, 1.5]) * BAR_W

# 2. Create a single grid (Rows = Strategies, Columns = Metrics)
n_rows, n_cols = len(strategies), len(METRICS)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows), squeeze=False, sharex=True)

for i, strategy in enumerate(strategies):
    # Set the index to frac_req once per strategy to eliminate nested list comprehension loops
    sub = df_s[df_s["sampling_strategy"] == strategy].set_index("frac_req")
    
    for j, metric in enumerate(METRICS):
        ax = axes[i, j]
        if sub.empty: continue

        # Plot bars
        for (suffix, label, color), offset in zip(METHODS, OFFSETS):
            col_mean = f"{metric}_{suffix}_mean"
            if col_mean in sub.columns:
                # reindex ensures all fracs are aligned, missing fracs get 0.0
                vals = sub[col_mean].reindex(fracs, fill_value=0.0).values
                ax.bar(x + offset, vals, width=BAR_W, color=color, label=label)

        # 3. Handle labeling cleanly to avoid redundancy
        if i == 0: # Top row: Metric titles
            ax.set_title(METRIC_LABELS[metric], fontsize=13, fontweight="bold")
        if j == 0: # Left column: Strategy names
            ax.set_ylabel(f"Strategy: {strategy}\n\nScore", fontsize=11, fontweight="bold")
        
        if i == n_rows - 1: # Bottom row: X-axis labels
            ax.set_xlabel("Fraction Required", fontsize=11)
            ax.set_xticks(x)
            ax.set_xticklabels([f"{f:.1f}" for f in fracs], rotation=45, fontsize=9)

        ax.set_ylim(0, None)
        ax.grid(axis="y", alpha=0.3)

# 4. Create a single, shared legend at the bottom of the figure
handles = [mpatches.Patch(color=c, label=lbl) for _, lbl, c in METHODS]
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0), 
           ncol=4, fontsize=10, title="Methods", title_fontsize=11)

plt.tight_layout()

# 5. Display the plot instead of saving it
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import scienceplots

# Apply SciencePlots style
plt.style.use(["science", "no-latex"])

CSV_PATH_S = "../experiments/sachs/b-load_results_new.csv"

# 1. Keep only F1 and Int. Distance
METRICS = ["f1", "int_dist"]
METRIC_LABELS = {"f1": "F1 Score", "int_dist": "Int. Distance"}

METHODS = [
    ("with_b",    "b-LOAD", "#1f77b4"),
    ("without_b", "LOAD",   "#aec7e8"),
    ("bpc",       "b-PC",   "#d62728"),
    ("pc",        "PC",     "#f4a582"),
]

# Load and clean data
df_s = pd.read_csv(CSV_PATH_S, sep=None, engine="python")
df_s.columns = df_s.columns.str.strip()

strategies = sorted(df_s["sampling_strategy"].unique())
fracs      = sorted(df_s["frac_req"].unique())
x          = np.arange(len(fracs))

BAR_W   = 0.18
OFFSETS = np.array([-1.5, -0.5, 0.5, 1.5]) * BAR_W

# Increased figsize to support the massive fonts without getting squished
n_rows, n_cols = len(strategies), len(METRICS)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 6 * n_rows), squeeze=False, sharex=True)

for i, strategy in enumerate(strategies):
    # Set the index to frac_req once per strategy to eliminate nested list comprehension loops
    sub = df_s[df_s["sampling_strategy"] == strategy].set_index("frac_req")
    
    for j, metric in enumerate(METRICS):
        ax = axes[i, j]
        if sub.empty: continue

        # Plot bars
        for (suffix, label, color), offset in zip(METHODS, OFFSETS):
            col_mean = f"{metric}_{suffix}_mean"
            if col_mean in sub.columns:
                # reindex ensures all fracs are aligned, missing fracs get 0.0
                vals = sub[col_mean].reindex(fracs, fill_value=0.0).values
                ax.bar(x + offset, vals, width=BAR_W, color=color, label=label)

        # Handle massive labeling cleanly
        if i == 0: # Top row: Metric titles
            ax.set_title(METRIC_LABELS[metric], fontsize=26, fontweight="bold", pad=15)
            
        if j == 0: # Left column: Strategy names
            ax.set_ylabel(f"Strategy: {strategy}\n\nScore", fontsize=24, fontweight="bold", labelpad=12)
        
        if i == n_rows - 1: # Bottom row: X-axis labels
            ax.set_xlabel("Fraction Required", fontsize=24, fontweight="bold", labelpad=10)
            ax.set_xticks(x)
            # Increased tick label size and kept rotation
            ax.set_xticklabels([f"{f:.1f}" for f in fracs], rotation=45, fontsize=18)

        # Tick parameters for Y-axis
        ax.tick_params(axis='y', which='major', labelsize=18)
        ax.set_ylim(0, None)
        ax.grid(axis="y", alpha=0.3, linestyle='--')

# Apply tight_layout BEFORE adding the external legend to lock the grid in place
plt.tight_layout(w_pad=2.0, h_pad=2.0)

# Increased top to 0.88 to let the subplots stretch further up, reducing the empty margin
fig.subplots_adjust(top=0.88)

# Brought the y-coordinate down from 0.92 to 0.90 to hug the top of the graphs
handles = [mpatches.Patch(color=c, label=lbl) for _, lbl, c in METHODS]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.90), 
           ncol=4, frameon=False, prop={'size': 24, 'weight': 'bold'}, columnspacing=2.0)

# Save the plot directly to PDF
plt.savefig("sachs_results_bar.pdf", bbox_inches="tight")
plt.show()

In [ ]:
df_sachs = pd.read_csv(CSV_PATH_S)

total_queries = 46

methods = {
    'LOAD': 'without_b',
    'b-LOAD (global sampling)': ('with_b', 'global'),
    'b-LOAD (local sampling)': ('with_b', 'local'),
    'PC': 'pc',
    'b-PC (global sampling)': ('bpc', 'global'),
    'b-PC (local sampling)': ('bpc', 'local'),
}

results = []

for method_name, method_info in methods.items():
    if isinstance(method_info, tuple):
        suffix, strategy = method_info
        sub_df = df_sachs[df_sachs['sampling_strategy'] == strategy].copy()
    else:
        suffix = method_info
        sub_df = df_sachs.copy()
        # Filter to just one strategy to avoid double counting rows for methods independent of strategy
        sub_df = sub_df[sub_df['sampling_strategy'] == 'local']
        
    queries_identified_list = []
    f1_list = []
    int_dist_list = []
    
    for frac, grp in sub_df.groupby('frac_req'):
        # For Sachs, there's only 1 row per frac_req (no subsets)
        queries = grp[f'recall_{suffix}_mean'].values[0] * total_queries
        queries_identified_list.append(queries)
        f1_list.append(grp[f'f1_{suffix}_mean'].values[0])
        int_dist_list.append(grp[f'int_dist_{suffix}_mean'].values[0])
        
    avg_queries = sum(queries_identified_list) / len(queries_identified_list)
    avg_f1 = sum(f1_list) / len(f1_list)
    avg_int_dist = sum(int_dist_list) / len(int_dist_list)
    
    results.append({
        'Method': method_name,
        'Queries identified': f"{avg_queries:.0f} / {total_queries}",
        'F1 (adj. set)': f"{avg_f1:.2f}",
        'Intervention Dist.': f"{avg_int_dist:.2f}"
    })

res_df = pd.DataFrame(results)
res_df

## Sanity check: intervention distance  

In [ ]:
from src.data_generation import generate_data

test_data = generate_data(
    seed=42,
    observed=10,
    exp_degree=2,
    max_degree=5,
    targets=2,
    identifiable=True,
    samples_num=1000
)

test_data

In [ ]:
nx.to_numpy_array(test_data["true_dag"])